# 🔑 Solutions: Lesson 15 · Keras & TensorFlow Bridge

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 15.1: Keras MLP factory  &nbsp; `⭐`

Write `keras_mlp(sizes)` returning a `keras.Sequential` for layer sizes like `[784, 128, 64, 10]`: an `Input((sizes[0],))`, then `Dense` layers with ReLU between and **no activation on the last** (logits).


In [2]:
import keras

def keras_mlp(sizes):
    layers = [keras.layers.Input((sizes[0],))]
    for i, n in enumerate(sizes[1:]):
        layers.append(keras.layers.Dense(n, activation="relu" if i < len(sizes) - 2 else None))
    return keras.Sequential(layers)

I0000 00:00:1794724293.936960    1251 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794724293.976097    1251 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1794724295.169447    1251 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
# ▶ self-check
m = keras_mlp([784, 128, 64, 10])
dl.check("109,386 parameters (as in Lessons 02 and 05)", m is not None and m.count_params() == 109386)
dl.check("output shape (4, 10)", m is not None and tuple(m(np.zeros((4, 784), "float32")).shape) == (4, 10))
dl.check("last layer has no activation", m is not None and m.layers[-1].activation.__name__ == "linear")

✅ 109,386 parameters (as in Lessons 02 and 05)
✅ output shape (4, 10)
✅ last layer has no activation


---
### 🧩 Problem 15.2: Translate the leaf CNN to Keras  &nbsp; `⭐⭐`

In Lesson 10 the PyTorch leaf classifier was three blocks of `Conv2d(3×3, padding=1) → BatchNorm → ReLU → MaxPool(2)` with 16, 32, 64 filters, then `Flatten → Dropout(0.3) → Linear(64·6·6 → 5)`.
Write `keras_leaf_cnn()` for **channels-last 48×48×3** inputs, returning a Keras model with the same architecture. Its *trainable* parameter count must equal PyTorch's **35,333**.


In [4]:
import keras

def keras_leaf_cnn():
    layers = [keras.layers.Input((48, 48, 3))]
    for f in (16, 32, 64):
        layers += [keras.layers.Conv2D(f, 3, padding="same"), keras.layers.BatchNormalization(),
                   keras.layers.ReLU(), keras.layers.MaxPooling2D(2)]
    layers += [keras.layers.Flatten(), keras.layers.Dropout(0.3), keras.layers.Dense(5)]
    return keras.Sequential(layers)

In [5]:
# ▶ self-check
m = keras_leaf_cnn()
n_trainable = sum(int(np.prod(w.shape)) for w in m.trainable_weights) if m is not None else None
dl.check("input is channels-last 48×48×3", m is not None and tuple(m.input_shape) == (None, 48, 48, 3))
dl.check("trainable parameters = 35,333 (same as PyTorch)", n_trainable == 35333, f"yours: {n_trainable}")
dl.check("outputs 5 logits", m is not None and tuple(m(np.zeros((2, 48, 48, 3), 'float32')).shape) == (2, 5))

✅ input is channels-last 48×48×3
✅ trainable parameters = 35,333 (same as PyTorch)
✅ outputs 5 logits


**Why this works.** BatchNorm has 2 trainable parameters per channel (γ, β) plus 2 *non-trainable* running statistics; Keras' `count_params()` includes the latter, so we compare trainable weights.

---
### 🧩 Problem 15.3: Channels first ↔ channels last  &nbsp; `⭐`

Write `to_channels_last(x)` converting `(N, C, H, W)` → `(N, H, W, C)` and `to_channels_first(x)` for the reverse, for NumPy arrays. (Remember Lesson 10's `transpose`.)


In [6]:
def to_channels_last(x):
    return x.transpose(0, 2, 3, 1)

def to_channels_first(x):
    return x.transpose(0, 3, 1, 2)

In [7]:
# ▶ self-check
x = np.random.default_rng(0).normal(size=(2, 3, 8, 6))
a = to_channels_last(x)
dl.check("(2,3,8,6) → (2,8,6,3)", a is not None and a.shape == (2, 8, 6, 3))
dl.check("round trip restores the original", a is not None and np.array_equal(to_channels_first(a), x))
dl.check("pixel (n=1, c=2, h=5, w=4) is preserved", a is not None and a[1, 5, 4, 2] == x[1, 2, 5, 4])

✅ (2,3,8,6) → (2,8,6,3)
✅ round trip restores the original
✅ pixel (n=1, c=2, h=5, w=4) is preserved


---
### 🧩 Problem 15.4: A step-decay schedule function  &nbsp; `⭐`

Keras' `LearningRateScheduler` takes a function `schedule(epoch, lr)` returning the learning rate for that epoch. Write `step_decay(epoch, lr)` that **halves** the learning rate at the start of epochs 5, 10, 15, … (i.e. when `epoch > 0 and epoch % 5 == 0`) and otherwise returns `lr` unchanged.


In [8]:
def step_decay(epoch, lr):
    return lr * 0.5 if epoch > 0 and epoch % 5 == 0 else lr

In [9]:
# ▶ self-check
lr, seq = 0.1, []
for e in range(12):
    lr = step_decay(e, lr) or lr
    seq.append(round(lr, 4))
dl.check("0.1 for epochs 0–4, 0.05 for 5–9, 0.025 for 10–11", seq == [0.1] * 5 + [0.05] * 5 + [0.025] * 2, f"got {seq}")

✅ 0.1 for epochs 0–4, 0.05 for 5–9, 0.025 for 10–11


---
### 🧩 Problem 15.5: A TensorFlow training step  &nbsp; `⭐⭐`

Write `train_step(model, loss_fn, optimizer, xb, yb)` using `tf.GradientTape`: compute the loss with `model(xb, training=True)`, get gradients, apply them, and return the loss as a Python float.
*(Needs TensorFlow; the self-check says so if it isn't installed.)*


In [10]:
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
try:
    import tensorflow as tf, keras
except ImportError:
    tf = None

def train_step(model, loss_fn, optimizer, xb, yb):
    with tf.GradientTape() as tape:
        loss = loss_fn(yb, model(xb, training=True))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return float(loss)

In [11]:
# ▶ self-check
if tf is None:
    print("⚠️ TensorFlow not installed: skipping this check (pip install tensorflow to enable it).")
else:
    keras.utils.set_random_seed(0)
    m = keras.Sequential([keras.layers.Input((4,)), keras.layers.Dense(16, activation="relu"), keras.layers.Dense(3)])
    rng = np.random.default_rng(0); xb = rng.normal(size=(64, 4)).astype("float32"); yb = (xb[:, 0] > 0).astype("int32") + (xb[:, 1] > 0)
    opt, lf = keras.optimizers.Adam(0.05), keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    losses = [train_step(m, lf, opt, xb, yb) for _ in range(40)]
    dl.check("returns a float", isinstance(losses[0], float))
    dl.check("loss decreases over 40 steps", losses[-1] < 0.6 * losses[0], f"{losses[0]:.3f} → {losses[-1]:.3f}")

✅ returns a float
✅ loss decreases over 40 steps


**Why this works.** Gradient tapes replace `zero_grad()`/`backward()`: the tape is created fresh inside the `with` block, so nothing accumulates between steps.

---
### 🧩 Problem 15.6: Ship it: export to TFLite and verify (challenge)  &nbsp; `⭐⭐⭐`

Write `to_tflite(keras_model, quantise=False)` returning the TFLite model as **bytes**: export the Keras model to a temporary SavedModel directory with `model.export(dir)`, then convert with `tf.lite.TFLiteConverter.from_saved_model(dir)`; if `quantise`, set `converter.optimizations = [tf.lite.Optimize.DEFAULT]`.
The check verifies the quantised file is smaller and still agrees with the Keras model.


In [12]:
import os, tempfile
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
try:
    import tensorflow as tf, keras
except ImportError:
    tf = None

def to_tflite(keras_model, quantise=False):
    d = tempfile.mkdtemp(); keras_model.export(d)
    conv = tf.lite.TFLiteConverter.from_saved_model(d)
    if quantise:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    return conv.convert()

In [13]:
# ▶ self-check
if tf is None:
    print("⚠️ TensorFlow not installed: skipping this check.")
else:
    import warnings; warnings.filterwarnings("ignore")
    keras.utils.set_random_seed(0)
    m = keras.Sequential([keras.layers.Input((28, 28, 1)), keras.layers.Conv2D(8, 5, padding="same", activation="relu"),
                          keras.layers.MaxPooling2D(2), keras.layers.Flatten(), keras.layers.Dense(10)])
    big, small = to_tflite(m), to_tflite(m, quantise=True)
    dl.check("returns bytes", isinstance(big, (bytes, bytearray)) and len(big) > 1000)
    dl.check(f"quantised model is smaller ({len(small) / 1024:.0f} KB vs {len(big) / 1024:.0f} KB)", len(small) < 0.6 * len(big))
    x = np.random.default_rng(0).random((20, 28, 28, 1)).astype("float32")
    def run(blob):
        it = tf.lite.Interpreter(model_content=blob); it.allocate_tensors(); i, o = it.get_input_details()[0], it.get_output_details()[0]
        outs = []
        for im in x: it.set_tensor(i["index"], im[None]); it.invoke(); outs.append(it.get_tensor(o["index"])[0])
        return np.array(outs)
    ref = m.predict(x, verbose=0)
    dl.check("float32 TFLite matches Keras", np.allclose(run(big), ref, atol=1e-4))
    dl.check("quantised TFLite is still close", np.abs(run(small) - ref).max() < 0.25 * (np.abs(ref).max() + 1e-6) + 0.05)

INFO:tensorflow:Assets written to: /tmp/tmpilzj8t81/assets


INFO:tensorflow:Assets written to: /tmp/tmpilzj8t81/assets


Saved artifact at '/tmp/tmpilzj8t81'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor_23')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  139739258240080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139739258238928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139739258234896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139739258238160: TensorSpec(shape=(), dtype=tf.resource, name=None)


INFO:tensorflow:Assets written to: /tmp/tmpn_w17fak/assets


W0000 00:00:1794724297.461327    1251 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794724297.461371    1251 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794724297.461921    1251 reader.cc:83] Reading SavedModel from: /tmp/tmpilzj8t81
I0000 00:00:1794724297.462351    1251 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794724297.462357    1251 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpilzj8t81
I0000 00:00:1794724297.464538    1251 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1794724297.465004    1251 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794724297.482026    1251 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpilzj8t81
I0000 00:00:1794724297.486233    1251 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 24319 microseconds.
I0000 00:00:1794724297.493157    1251

Saved artifact at '/tmp/tmpn_w17fak'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor_23')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  139739258240080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139739258238928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139739258234896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139739258238160: TensorSpec(shape=(), dtype=tf.resource, name=None)


✅ returns bytes
✅ quantised model is smaller (19 KB vs 65 KB)
✅ float32 TFLite matches Keras
✅ quantised TFLite is still close


W0000 00:00:1794724297.907612    1251 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794724297.907659    1251 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794724297.907851    1251 reader.cc:83] Reading SavedModel from: /tmp/tmpn_w17fak
I0000 00:00:1794724297.908263    1251 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794724297.908269    1251 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpn_w17fak
I0000 00:00:1794724297.910790    1251 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794724297.923546    1251 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpn_w17fak
I0000 00:00:1794724297.927872    1251 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 20028 microseconds.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


**Why this works.** Quantisation trades a few bits of precision for a much smaller, faster model: on classification tasks the argmax usually survives intact.